In [6]:
import os
import random
from contextlib import nullcontext
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

# ============================================================
# 0. CONFIGURATION
# ============================================================
CSV_PATH = "KO.csv"

RESULTS_CSV = "rnn_quantile_grid_results.csv"
BEST_PRED_CSV ="rnn_best_predictions.csv"

TARGET_COVERAGE = 0.95
RANDOM_SEED = 42

LOOKBACK_LIST = [30, 60, 90,120]
LAG_LIST = [1, 3, 5,10]
LAYER_LIST = [1, 2]
HIDDEN_LIST = [64, 128]
P_LIST = [(0.05, 0.95), (0.025, 0.975)]

BATCH_SIZE = 64
EPOCHS = 200
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 6
GRAD_CLIP = 1.0
DROPOUT = 0.20

print("============================================================")
print("USING MODEL: RNN")
print("LOOKBACK LIST:", LOOKBACK_LIST)
print("LAG LIST:", LAG_LIST)
print("LAYER LIST:", LAYER_LIST)
print("HIDDEN LIST:", HIDDEN_LIST)
print("P LIST:", P_LIST)
print("TARGET COVERAGE:", TARGET_COVERAGE)
print("============================================================")

# ============================================================
# 1. DEVICE + UPDATED AMP (FIXED)
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"

def autocast_ctx():
    return torch.amp.autocast("cuda") if use_amp else nullcontext()

scaler_amp = torch.amp.GradScaler("cuda") if use_amp else None

print("DEVICE:", device)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)

# ============================================================
# 2. LOAD DATA + FEATURE ENGINEERING
# ============================================================
df = pd.read_csv(CSV_PATH)
df.columns = df.columns.str.strip()

if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date").reset_index(drop=True)

df["log_return"] = np.log(df["Close"]).diff()
df["volatility"] = df["log_return"].rolling(5).std()
df["momentum_5"] = df["Close"] - df["Close"].shift(5)
df["ma_5"] = df["Close"].rolling(5).mean()
df["ma_10"] = df["Close"].rolling(10).mean()
df["vol_change"] = df["Volume"].pct_change()
df["range_hl"] = (df["High"] - df["Low"]) / df["Close"]

if "Scaled_sentiment" in df.columns:
    df["sentiment_signed"] = 2 * df["Scaled_sentiment"] - 1
else:
    df["sentiment_signed"] = 0.0

if "News_flag" not in df.columns:
    df["News_flag"] = 0

df["sentiment_active"] = df["sentiment_signed"] * df["News_flag"]
df["sentiment_impact"] = df["sentiment_active"] * df["log_return"].shift(1)

df["target"] = df["Close"].shift(-1)
df = df.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)

# ============================================================
# 3. FEATURE SETS
# ============================================================
BASE_FEATURES = [
    "Close","log_return","volatility","momentum_5",
    "ma_5","ma_10","Volume","vol_change","range_hl"
]

FEATURE_SETS = {
    "non_semantic": BASE_FEATURES,
    "semantic": BASE_FEATURES + ["News_flag","sentiment_signed","sentiment_active","sentiment_impact"]
}

print("\nFEATURE SETS USED:")
print("Non-semantic:", NON_SEMANTIC_FEATURES)
print("Semantic:", SEMANTIC_FEATURES)

# ============================================================
# 4. HELPERS
# ============================================================
def create_lag_features(dataframe, base_cols, n_lags):
    out = dataframe.copy()
    lag_blocks = []
    for k in range(1, n_lags + 1):
        lag_blocks.append(out[base_cols].shift(k).add_suffix(f"_lag{k}"))
    out = pd.concat([out] + lag_blocks, axis=1)
    out = out.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
    return out

def create_sequences(dataframe, feature_cols, lookback):
    X, y, dates = [], [], []
    X_arr = dataframe[feature_cols].to_numpy(dtype=np.float32, copy=False)
    y_arr = dataframe["target"].to_numpy(dtype=np.float32, copy=False)
    date_arr = dataframe["Date"].to_numpy() if "Date" in dataframe.columns else np.arange(len(dataframe))

    for i in range(lookback, len(dataframe)):
        X.append(X_arr[i - lookback:i])
        y.append(y_arr[i])
        dates.append(date_arr[i])

    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32), np.asarray(dates)

def chronological_split(X, y, dates, train_frac=0.70, val_frac=0.15):
    n = len(X)
    train_end = int(n * train_frac)
    val_end = int(n * (train_frac + val_frac))
    return {
        "train": (X[:train_end], y[:train_end], dates[:train_end]),
        "val":   (X[train_end:val_end], y[train_end:val_end], dates[train_end:val_end]),
        "test":  (X[val_end:], y[val_end:], dates[val_end:]),
    }

def make_loader(X, y=None, batch_size=64, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    if y is None:
        ds = TensorDataset(X_t)
    else:
        y_t = torch.tensor(y, dtype=torch.float32)
        ds = TensorDataset(X_t, y_t)

    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        pin_memory=(device.type == "cuda"),
        drop_last=False
    )

# ============================================================
# 5. RNN MODEL
# ============================================================
class RNNQuantileModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=DROPOUT):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            nonlinearity="tanh",
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.norm = nn.LayerNorm(hidden_size)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        last = out[:, -1, :]
        last = self.norm(last)
        last = self.dropout(last)
        return self.head(last).squeeze(-1)

# ============================================================
# 6. LOSS
# ============================================================
def pinball_loss(pred, target, q):
    e = target - pred
    return torch.mean(torch.maximum(q * e, (q - 1.0) * e))

# ============================================================
# 7. TRAINING / PREDICTION
# ============================================================
def train_quantile_model(X_train, y_train, X_val, y_val, q, input_size, hidden_size, num_layers):
    model = RNNQuantileModel(input_size, hidden_size, num_layers).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    train_loader = make_loader(X_train, y_train, batch_size=BATCH_SIZE, shuffle=False)
    val_loader = make_loader(X_val, y_val, batch_size=BATCH_SIZE, shuffle=False)

    best_state = None
    best_val = float("inf")
    patience_count = 0

    for epoch in range(EPOCHS):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with autocast_ctx():
                pred = model(xb)
                loss = pinball_loss(pred, yb, q)

            if use_amp:
                scaler_amp.scale(loss).backward()
                scaler_amp.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler_amp.step(optimizer)
                scaler_amp.update()
            else:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()

            train_losses.append(loss.detach().item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)
                with autocast_ctx():
                    pred = model(xb)
                    vloss = pinball_loss(pred, yb, q)
                val_losses.append(vloss.item())

        mean_val = float(np.mean(val_losses))
        if mean_val < best_val - 1e-6:
            best_val = mean_val
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, best_val

@torch.no_grad()
def predict_model(model, X):
    model.eval()
    loader = make_loader(X, y=None, batch_size=512, shuffle=False)
    preds = []
    for (xb,) in loader:
        xb = xb.to(device, non_blocking=True)
        with autocast_ctx():
            out = model(xb)
        preds.append(out.detach().float().cpu().numpy())
    return np.concatenate(preds, axis=0)

# ============================================================
# 8. CALIBRATION FOR 95% COVERAGE WITH MINIMAL WIDTH
# ============================================================
CALIBRATION_GRID = np.linspace(0.50, 2.50, 81)

def interval_metrics(y_true, low, high):
    low = np.asarray(low)
    high = np.asarray(high)
    y_true = np.asarray(y_true)
    coverage = np.mean((y_true >= low) & (y_true <= high))
    width = np.mean(high - low)
    mid = 0.5 * (low + high)
    mse = mean_squared_error(y_true, mid)
    mae = mean_absolute_error(y_true, mid)
    return coverage, width, mse, mae

def calibrate_interval(y_val, low_val, high_val, target_coverage=TARGET_COVERAGE):
    mid = 0.5 * (np.asarray(low_val) + np.asarray(high_val))
    half = 0.5 * (np.asarray(high_val) - np.asarray(low_val))
    half = np.maximum(half, 1e-8)

    best = None
    for s in CALIBRATION_GRID:
        low_s = mid - s * half
        high_s = mid + s * half
        cov, width, _, _ = interval_metrics(y_val, low_s, high_s)

        # prioritize target coverage, then smaller width
        score = (cov < target_coverage, abs(cov - target_coverage), width)
        if best is None or score < best["score"]:
            best = {
                "scale": float(s),
                "score": score,
                "coverage": float(cov),
                "width": float(width),
            }
    return best["scale"], best

def apply_scale(low, high, scale):
    mid = 0.5 * (np.asarray(low) + np.asarray(high))
    half = 0.5 * (np.asarray(high) - np.asarray(low))
    return mid - scale * half, mid + scale * half

# ============================================================
# 9. GRID SEARCH
# ============================================================
results = []
artifacts = {}

for feature_set_name, feature_cols in FEATURE_SETS.items():
    print(f"\n==================== {feature_set_name.upper()} ====================")

    for n_lags in LAG_LIST:
        lagged_df = create_lag_features(df, feature_cols, n_lags)
        lagged_feature_cols = feature_cols + [f"{c}_lag{k}" for c in feature_cols for k in range(1, n_lags + 1)]

        for lookback in LOOKBACK_LIST:
            X_seq, y_seq, d_seq = create_sequences(lagged_df, lagged_feature_cols, lookback)
            if len(X_seq) < 100:
                continue

            splits = chronological_split(X_seq, y_seq, d_seq, train_frac=0.70, val_frac=0.15)
            X_train, y_train, d_train = splits["train"]
            X_val, y_val, d_val = splits["val"]
            X_test, y_test, d_test = splits["test"]

            # --- scaling ---
            x_scaler = StandardScaler()
            y_scaler = StandardScaler()

            X_train_2d = X_train.reshape(-1, X_train.shape[-1])
            X_val_2d = X_val.reshape(-1, X_val.shape[-1])
            X_test_2d = X_test.reshape(-1, X_test.shape[-1])

            x_scaler.fit(X_train_2d)
            y_scaler.fit(y_train.reshape(-1, 1))

            def scale_X(X):
                return x_scaler.transform(X.reshape(-1, X.shape[-1])).reshape(X.shape).astype(np.float32)

            X_train_s = scale_X(X_train)
            X_val_s = scale_X(X_val)
            X_test_s = scale_X(X_test)

            y_train_s = y_scaler.transform(y_train.reshape(-1, 1)).ravel().astype(np.float32)
            y_val_s = y_scaler.transform(y_val.reshape(-1, 1)).ravel().astype(np.float32)

            input_size = X_train_s.shape[-1]

            for num_layers in LAYER_LIST:
                for hidden_size in HIDDEN_LIST:
                    for q_low, q_high in P_LIST:
                        q_label = f"{q_low:.3f}/{q_high:.3f}"
                        print(
                            f"Training | set={feature_set_name} | lookback={lookback} | lag={n_lags} | "
                            f"layers={num_layers} | hidden={hidden_size} | p={q_label}"
                        )

                        # --- train quantile models ---
                        low_model, low_val_loss = train_quantile_model(
                            X_train_s, y_train_s, X_val_s, y_val_s,
                            q_low, input_size, hidden_size, num_layers
                        )
                        high_model, high_val_loss = train_quantile_model(
                            X_train_s, y_train_s, X_val_s, y_val_s,
                            q_high, input_size, hidden_size, num_layers
                        )

                        # --- raw predictions ---
                        low_val_s = predict_model(low_model, X_val_s)
                        high_val_s = predict_model(high_model, X_val_s)
                        low_test_s = predict_model(low_model, X_test_s)
                        high_test_s = predict_model(high_model, X_test_s)

                        low_val = y_scaler.inverse_transform(low_val_s.reshape(-1, 1)).ravel()
                        high_val = y_scaler.inverse_transform(high_val_s.reshape(-1, 1)).ravel()
                        low_test = y_scaler.inverse_transform(low_test_s.reshape(-1, 1)).ravel()
                        high_test = y_scaler.inverse_transform(high_test_s.reshape(-1, 1)).ravel()

                        # --- calibration scaling to target coverage ---
                        scale, calib_info = calibrate_interval(y_val, low_val, high_val, TARGET_COVERAGE)
                        low_test_cal, high_test_cal = apply_scale(low_test, high_test, scale)
                        low_val_cal, high_val_cal = apply_scale(low_val, high_val, scale)

                        # --- metrics ---
                        val_cov, val_width, val_mse, val_mae = interval_metrics(y_val, low_val_cal, high_val_cal)
                        test_cov, test_width, test_mse, test_mae = interval_metrics(y_test, low_test_cal, high_test_cal)

                        row = {
                            "model": "RNN",
                            "feature_set": feature_set_name,
                            "lookback": lookback,
                            "lag": n_lags,
                            "num_layers": num_layers,
                            "hidden_size": hidden_size,
                            "q_low": q_low,
                            "q_high": q_high,
                            "p_label": q_label,
                            "target_coverage": TARGET_COVERAGE,
                            "calibration_scale": scale,
                            "val_coverage": val_cov,
                            "val_width": val_width,
                            "val_mse": val_mse,
                            "val_mae": val_mae,
                            "test_coverage": test_cov,
                            "test_width": test_width,
                            "test_mse": test_mse,
                            "test_mae": test_mae,
                            "val_pinball_low": low_val_loss,
                            "val_pinball_high": high_val_loss,
                            "n_train": len(X_train_s),
                            "n_val": len(X_val_s),
                            "n_test": len(X_test_s),
                        }
                        results.append(row)

                        key = (
                            f"{feature_set_name}|L{lookback}|lag{n_lags}|"
                            f"{num_layers}x{hidden_size}|p{q_label}"
                        )
                        artifacts[key] = {
                            "dates_test": d_test,
                            "y_test": y_test,
                            "low_test": low_test_cal,
                            "high_test": high_test_cal,
                            "mid_test": 0.5 * (low_test_cal + high_test_cal),
                            "feature_set": feature_set_name,
                            "lookback": lookback,
                            "lag": n_lags,
                            "num_layers": num_layers,
                            "hidden_size": hidden_size,
                            "p_label": q_label,
                        }

results_df = pd.DataFrame(results)

# Rank by closeness to target coverage, then width, then error
results_df["coverage_gap"] = (results_df["test_coverage"] - TARGET_COVERAGE).abs()
results_df = results_df.sort_values(
    ["coverage_gap", "test_width", "test_mse", "test_mae"],
    ascending=[True, True, True, True]
).reset_index(drop=True)

results_df.to_csv(RESULTS_CSV, index=False)

print("\nTOP RESULTS")
print(results_df.head(10).to_string(index=False))

# Best config overall
best_row = results_df.iloc[0].to_dict()
best_key = (
    f"{best_row['feature_set']}|L{int(best_row['lookback'])}|lag{int(best_row['lag'])}|"
    f"{int(best_row['num_layers'])}x{int(best_row['hidden_size'])}|p{best_row['p_label']}"
)
best_art = artifacts[best_key]

best_pred_df = pd.DataFrame({
    "Date": pd.to_datetime(best_art["dates_test"]),
    "Actual_Close": best_art["y_test"],
    "Lower_PI": best_art["low_test"],
    "Upper_PI": best_art["high_test"],
    "Midpoint": best_art["mid_test"],
})
best_pred_df.to_csv(BEST_PRED_CSV, index=False)

print("\nBEST CONFIG")
print(pd.Series(best_row).to_string())

# ============================================================
# 10. OLD CHARTS KEPT + NEW COMPARISON CHARTS ADDED
# ============================================================

# -----------------------
# 10A. Prediction interval plot for best configuration
# -----------------------
plt.figure(figsize=(13, 5))
plt.plot(best_art["dates_test"], best_art["y_test"], label="Actual Close", linewidth=1.5)
plt.plot(best_art["dates_test"], best_art["mid_test"], label="Midpoint", linewidth=1.2)
plt.fill_between(best_art["dates_test"], best_art["low_test"], best_art["high_test"], alpha=0.25, label="Prediction Interval")
plt.title(
    f"Best Forecast | {best_row['feature_set']} | L={int(best_row['lookback'])} | lag={int(best_row['lag'])} | "
    f"{int(best_row['num_layers'])} layers | {int(best_row['hidden_size'])} neurons | p={best_row['p_label']}"
)
plt.xlabel("Date")
plt.ylabel("Close Price")
plt.legend()
plt.tight_layout()
plt.show()

# -----------------------
# 10B. Lag vs Width / Coverage for semantic and non-semantic
# -----------------------
for metric in ["test_width", "test_coverage"]:
    plt.figure(figsize=(12, 5))
    for feature_set_name in results_df["feature_set"].unique():
        sub = results_df[results_df["feature_set"] == feature_set_name].groupby("lag", as_index=False)[metric].mean()
        plt.plot(sub["lag"], sub[metric], marker="o", linewidth=2, label=feature_set_name)
    plt.title(f"Lag vs {metric.replace('_', ' ').title()} (Semantic vs Non-Semantic)")
    plt.xlabel("Lag")
    plt.ylabel(metric.replace("_", " ").title())
    plt.legend()
    plt.tight_layout()
    plt.show()

# -----------------------
# 10C. p-value comparison plots for both feature sets
# -----------------------
for metric in ["test_width", "test_coverage"]:
    plt.figure(figsize=(12, 5))
    for feature_set_name in results_df["feature_set"].unique():
        sub = results_df[results_df["feature_set"] == feature_set_name].groupby("p_label", as_index=False)[metric].mean()
        # preserve order in P_LIST
        order = [f"{a:.3f}/{b:.3f}" for a, b in P_LIST]
        sub["p_label"] = pd.Categorical(sub["p_label"], categories=order, ordered=True)
        sub = sub.sort_values("p_label")
        plt.plot(sub["p_label"], sub[metric], marker="o", linewidth=2, label=feature_set_name)
    plt.title(f"p-value vs {metric.replace('_', ' ').title()} (Semantic vs Non-Semantic)")
    plt.xlabel("p-value pair")
    plt.ylabel(metric.replace("_", " ").title())
    plt.legend()
    plt.tight_layout()
    plt.show()

# -----------------------
# 10D. Lookback sequence charts
# -----------------------
for metric in ["test_width", "test_coverage", "test_mse", "test_mae"]:
    plt.figure(figsize=(12, 5))
    for feature_set_name in results_df["feature_set"].unique():
        sub = results_df[results_df["feature_set"] == feature_set_name].groupby("lookback", as_index=False)[metric].mean()
        plt.plot(sub["lookback"], sub[metric], marker="o", linewidth=2, label=feature_set_name)
    plt.title(f"Lookback Sequence vs {metric.replace('_', ' ').title()} (Semantic vs Non-Semantic)")
    plt.xlabel("Lookback")
    plt.ylabel(metric.replace("_", " ").title())
    plt.legend()
    plt.tight_layout()
    plt.show()

# -----------------------
# 10E. Layer / hidden / lookback sensitivity plots
# -----------------------
for metric in ["test_width", "test_coverage"]:
    # Layers
    plt.figure(figsize=(12, 5))
    for feature_set_name in results_df["feature_set"].unique():
        sub = results_df[results_df["feature_set"] == feature_set_name].groupby("num_layers", as_index=False)[metric].mean()
        plt.plot(sub["num_layers"], sub[metric], marker="o", linewidth=2, label=feature_set_name)
    plt.title(f"Number of Layers vs {metric.replace('_', ' ').title()}")
    plt.xlabel("Number of Layers")
    plt.ylabel(metric.replace("_", " ").title())
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Hidden units
    plt.figure(figsize=(12, 5))
    for feature_set_name in results_df["feature_set"].unique():
        sub = results_df[results_df["feature_set"] == feature_set_name].groupby("hidden_size", as_index=False)[metric].mean()
        plt.plot(sub["hidden_size"], sub[metric], marker="o", linewidth=2, label=feature_set_name)
    plt.title(f"Hidden Units vs {metric.replace('_', ' ').title()}")
    plt.xlabel("Hidden Units")
    plt.ylabel(metric.replace("_", " ").title())
    plt.legend()
    plt.tight_layout()
    plt.show()

# -----------------------
# 10F. Comparison chart for semantic vs non-semantic
# -----------------------
compare_df = results_df.groupby("feature_set")[["test_coverage", "test_width", "test_mse", "test_mae"]].mean().reset_index()
print("\nSEMANTIC vs NON-SEMANTIC COMPARISON")
print(compare_df.to_string(index=False))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
metrics = ["test_coverage", "test_width", "test_mse", "test_mae"]
titles = ["Coverage Probability", "Mean Interval Width", "MSE", "MAE"]

for ax, metric, title in zip(axes.ravel(), metrics, titles):
    ax.bar(compare_df["feature_set"], compare_df[metric])
    ax.set_title(title)
    ax.set_xlabel("Feature Set")
    ax.set_ylabel(title)
    ax.grid(axis="y", alpha=0.25)

plt.suptitle("Semantic vs Non-Semantic Performance Comparison", y=1.02)
plt.tight_layout()
plt.show()

# -----------------------
# 10G. Trade-off plot: coverage vs width
# -----------------------
plt.figure(figsize=(12, 5))
for feature_set_name in results_df["feature_set"].unique():
    sub = results_df[results_df["feature_set"] == feature_set_name]
    plt.scatter(sub["test_width"], sub["test_coverage"], alpha=0.6, label=feature_set_name)
plt.axhline(TARGET_COVERAGE, linestyle="--", linewidth=1.2, color="black", label="Target Coverage")
plt.xlabel("Mean Interval Width")
plt.ylabel("Coverage Probability")
plt.title("Coverage vs Interval Width across all experiments")
plt.legend()
plt.tight_layout()
plt.show()

# ============================================================
# 11. SUMMARY PRINTS
# ============================================================
base_best = results_df[results_df["feature_set"] == "non_semantic"].iloc[0]
sem_best = results_df[results_df["feature_set"] == "semantic"].iloc[0]

print("\nBEST NON-SEMANTIC RESULT")
print(base_best[["test_coverage", "test_width", "test_mse", "test_mae"]].to_string())

print("\nBEST SEMANTIC RESULT")
print(sem_best[["test_coverage", "test_width", "test_mse", "test_mae"]].to_string())

print("\nDIFFERENCE (semantic - non_semantic)")
print(pd.Series({
    "coverage_diff": sem_best["test_coverage"] - base_best["test_coverage"],
    "width_diff": sem_best["test_width"] - base_best["test_width"],
    "mse_diff": sem_best["test_mse"] - base_best["test_mse"],
    "mae_diff": sem_best["test_mae"] - base_best["test_mae"],
}).to_string())

print(f"\nSaved results to: {RESULTS_CSV}")
print(f"Saved best predictions to: {BEST_PRED_CSV}")

USING MODEL: RNN
LOOKBACK LIST: [30, 60, 90, 120]
LAG LIST: [1, 3, 5, 10]
LAYER LIST: [1, 2]
HIDDEN LIST: [64, 128]
P LIST: [(0.05, 0.95), (0.025, 0.975)]
TARGET COVERAGE: 0.95
DEVICE: cuda

FEATURE SETS USED:
Non-semantic: ['Close', 'log_return', 'volatility', 'momentum_5', 'ma_5', 'ma_10', 'Volume', 'vol_change', 'range_hl']
Semantic: ['Close', 'log_return', 'volatility', 'momentum_5', 'ma_5', 'ma_10', 'Volume', 'vol_change', 'range_hl', 'News_flag', 'sentiment_signed', 'sentiment_active', 'sentiment_impact']

==================== NON_SEMANTIC ====================
Training | set=non_semantic | lookback=30 | lag=1 | layers=1 | hidden=64 | p=0.050/0.950


Training | set=non_semantic | lookback=30 | lag=1 | layers=1 | hidden=64 | p=0.025/0.975
Training | set=non_semantic | lookback=30 | lag=1 | layers=1 | hidden=128 | p=0.050/0.950
Training | set=non_semantic | lookback=30 | lag=1 | layers=1 | hidden=128 | p=0.025/0.975
Training | set=non_semantic | lookback=30 | lag=1 | layers=2 | hidden=64 | p=0.050/0.950
Training | set=non_semantic | lookback=30 | lag=1 | layers=2 | hidden=64 | p=0.025/0.975
Training | set=non_semantic | lookback=30 | lag=1 | layers=2 | hidden=128 | p=0.050/0.950
Training | set=non_semantic | lookback=30 | lag=1 | layers=2 | hidden=128 | p=0.025/0.975
Training | set=non_semantic | lookback=60 | lag=1 | layers=1 | hidden=64 | p=0.050/0.950
Training | set=non_semantic | lookback=60 | lag=1 | layers=1 | hidden=64 | p=0.025/0.975
Training | set=non_semantic | lookback=60 | lag=1 | layers=1 | hidden=128 | p=0.050/0.950
Training | set=non_semantic | lookback=60 | lag=1 | layers=1 | hidden=128 | p=0.025/0.975
Training | set=

OutOfMemoryError: CUDA out of memory. Tried to allocate 106.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 26.81 MiB is free. Process 4032 has 5.54 GiB memory in use. Process 71291 has 8.68 GiB memory in use. Including non-PyTorch memory, this process has 320.00 MiB memory in use. Of the allocated memory 58.64 MiB is allocated by PyTorch, and 119.36 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)